# HVAC Prediction Workflow

This notebook provides a streamlined workflow for training, optimizing, and ensembling an LSTM model for HVAC power consumption prediction.

**Workflow:**
1.  **Setup:** Import libraries, define configuration, helper functions, dataset, and model architecture.
2.  **Data Preparation:** Load, preprocess, scale data, and create PyTorch DataLoaders.
3.  **Hyperparameter Optimization (Optuna):** Tune model hyperparameters to find the best combination based on validation performance.
4.  **Ensemble Training (Bagging):** Train multiple models using the best hyperparameters found, each with a different random seed.
5.  **Ensemble Prediction:** Use the trained ensemble to make predictions on the test set and generate a submission file.


## 1. Setup

Import necessary libraries, define configuration parameters, utility functions, the dataset class, and the model architecture.

In [71]:
# ! pip install joblib
# ! pip install tqdm
# ! pip install torch
# ! pip install pandas
# ! pip install numpy
# ! pip install matplotlib
# ! pip install optuna
# ! pip install scikit-learn

In [72]:
# --- Imports ---
import os
import json
import random
import joblib
import pprint
import argparse # Used for Namespace
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import optuna
from tqdm import tqdm # Use notebook-friendly tqdm

In [73]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [74]:
# --- Configuration ---
# Simplified configuration using a dictionary
config = {
    # Data/Paths
    "train_csv": "dataset/train_split.csv",
    "test_csv": "dataset/test_split.csv", # Used for evaluation during training/optimization
    "predict_features_csv": "dataset/test_features.csv", # Used for final prediction
    "output_dir": "models_lstm_simplified", # Base directory for saving models/scalers
    "results_dir": "results_lstm_simplified", # Base directory for saving results/submissions
    "base_feature_cols": [
        'temperature_exterieure', 'humidite', 'ensoleillement',
        'vitesse_vent', 'direction_vent', 'temperature_interieure'
    ],
    "target_col": 'puissance_cvac_future',

    # Model Defaults (can be overridden by Optuna)
    "seq_len": 32,
    "hidden_size": 128,
    "num_layers": 2,
    "dropout": 0.2,

    # Training Defaults (can be overridden by Optuna)
    "batch_size": 64,
    "num_epochs": 50, # Max epochs for a single trial
    "lr": 1e-3,
    "loss_function": "MSE", # 'MSE' or 'Huber'
    "val_split_ratio": 0.15,

    # Feature Engineering
    "use_time_features": True,

    # Training Enhancements
    "use_lr_scheduler": True,
    "lr_scheduler_patience": 5,
    "lr_scheduler_factor": 0.5,
    "use_early_stopping": True,
    "early_stopping_patience": 10,
    "use_gradient_clipping": True,
    "gradient_clip_max_norm": 1.0,

    # Optimization (Optuna)
    "n_trials": 10, # Number of Optuna trials
    "optuna_study_name": "hvac_optimization_lstm_simplified",
    "optuna_storage_url": None, # e.g., "sqlite:///hvac_optuna_lstm_simplified.db"

    # Ensemble (Bagging)
    "ensemble_seeds": [42, 123, 456, 789, 1011], # Seeds for ensemble members
    "ensemble_artifacts_file": "ensemble_artifacts.json", # File to store paths for ensemble models
    "final_submission_filename": "submission_ensemble.csv",
}


In [75]:
# --- Utility Functions ---
def add_time_features(df):
    """Adds time-based features to the DataFrame."""
    if 'date' not in df.columns:
        raise ValueError("'date' column is required but not found.")
    df['date'] = pd.to_datetime(df['date'])
    df['hour'] = df['date'].dt.hour
    df['dayofweek'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24.0)
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12.0)
    df['month_cos'] = np.cos(2 * np.pi * df['month']/12.0)
    return df

def set_seed(seed):
    """Sets random seed for reproducibility."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [76]:
# --- Core Components ---
class HVACDataset(Dataset):
    """PyTorch Dataset for HVAC sequences."""
    def __init__(self, sequences, targets):
        self.X = sequences.astype(np.float32)
        self.y = targets.astype(np.float32).reshape(-1, 1)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class LSTMModel(nn.Module):
    """LSTM Model for time series prediction."""
    def __init__(self, input_size, hidden_size, num_layers=1, dropout=0.0):
        super().__init__()
        lstm_dropout = dropout if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=lstm_dropout)
        self.bn = nn.BatchNorm1d(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        last_time_step_out = out[:, -1, :]
        bn_out = self.bn(last_time_step_out)
        dropout_out = self.dropout(bn_out)
        final_out = self.fc(dropout_out)
        return final_out

def create_sequences(features, consumption, seq_len):
    """Creates input sequences and targets."""
    X, y = [], []
    N = len(consumption)
    # Target is consumption 16 steps after the *end* of the input sequence
    target_offset = 15
    for i in range(N - seq_len - target_offset):
        X.append(features[i:i+seq_len])
        y.append(consumption[i + seq_len + target_offset])
    return np.array(X), np.array(y)

## 2. Data Preparation

Load the training and testing data, apply scaling, generate time features (optional), create sequences, split into train/validation sets, and prepare DataLoaders.

In [ ]:
def load_and_prepare_data(cfg):
    """Loads, preprocesses, and prepares data loaders."""
    print("Loading and preparing data...")
    
    # --- Load Data ---
    df_train_val = pd.read_csv(cfg['train_csv'], parse_dates=['date'])
    df_test = pd.read_csv(cfg['test_csv'], parse_dates=['date'])
    df_train_val = df_train_val.sort_values('date').reset_index(drop=True)
    df_test = df_test.sort_values('date').reset_index(drop=True)

    # --- Remove outliers ---
    df_train_val = df_train_val[df_train_val['puissance_cvac_future'] != 'missing']
    df_test = df_test[df_test['puissance_cvac_future'] != 'missing']

    # --- Feature Engineering ---
    all_feature_cols = list(cfg['base_feature_cols']) # Copy
    if cfg['use_time_features']:
        df_train_val = add_time_features(df_train_val)
        df_test = add_time_features(df_test)
        time_feature_cols = ['hour_sin', 'hour_cos', 'dayofweek', 'month_sin', 'month_cos']
        all_feature_cols.extend(time_feature_cols)

    # --- Add Temperature Difference Feature (Past 6 steps) ---
    temp_diff_lag = 6 # Look back 6 time steps (e.g., 6 hours if data is hourly)
    temp_diff_col_name = f'temp_diff_{temp_diff_lag}hr'
    df_train_val[temp_diff_col_name] = df_train_val['temperature_exterieure'].diff(periods=temp_diff_lag).fillna(0)
    df_test[temp_diff_col_name] = df_test['temperature_exterieure'].diff(periods=temp_diff_lag).fillna(0)
    all_feature_cols.append(temp_diff_col_name)

    # --- Add Rolling Temperature Statistics (Past 6 steps) ---
    rolling_window = 6
    temp_roll_mean_col = f'temp_roll_mean_{rolling_window}hr'
    temp_roll_std_col = f'temp_roll_std_{rolling_window}hr'
    df_train_val[temp_roll_mean_col] = df_train_val['temperature_exterieure'].rolling(window=rolling_window).mean().fillna(0)
    df_train_val[temp_roll_std_col] = df_train_val['temperature_exterieure'].rolling(window=rolling_window).std().fillna(0)
    df_test[temp_roll_mean_col] = df_test['temperature_exterieure'].rolling(window=rolling_window).mean().fillna(0)
    df_test[temp_roll_std_col] = df_test['temperature_exterieure'].rolling(window=rolling_window).std().fillna(0)
    all_feature_cols.extend([temp_roll_mean_col, temp_roll_std_col])


    feat_train_val = df_train_val[all_feature_cols].values
    targ_train_val = df_train_val[cfg['target_col']].values
    feat_test = df_test[all_feature_cols].values
    targ_test = df_test[cfg['target_col']].values

    print(f"Feature train_val shape: {feat_train_val.shape}")
    from IPython.display import display

    # Display the first few rows of the training and testing data
    print("Training Data Sample:")
    display(pd.DataFrame(feat_train_val, columns=all_feature_cols).head(12))

    print("Testing Data Sample:")
    display(pd.DataFrame(feat_test, columns=all_feature_cols).head())

    # --- Scaling ---
    feat_scaler = MinMaxScaler()
    feat_train_val_s = feat_scaler.fit_transform(feat_train_val)
    feat_test_s = feat_scaler.transform(feat_test)

    targ_scaler = MinMaxScaler()
    targ_train_val_s = targ_scaler.fit_transform(targ_train_val.reshape(-1,1)).flatten()
    targ_test_s = targ_scaler.transform(targ_test.reshape(-1,1)).flatten()

    # --- Create Sequences ---
    X_train_val, y_train_val = create_sequences(feat_train_val_s, targ_train_val_s, cfg['seq_len'])
    X_test, y_test = create_sequences(feat_test_s, targ_test_s, cfg['seq_len'])

    # --- Split Train/Validation ---
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=cfg['val_split_ratio'], shuffle=False
    )

    # --- DataLoaders ---
    train_ds = HVACDataset(X_train, y_train)
    val_ds = HVACDataset(X_val, y_val)
    test_ds = HVACDataset(X_test, y_test)

    train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True, pin_memory=True, num_workers=4)
    val_loader = DataLoader(val_ds, batch_size=cfg['batch_size'], shuffle=False, pin_memory=True, num_workers=4)
    test_loader = DataLoader(test_ds, batch_size=cfg['batch_size'], shuffle=False)

    print(f"Data loaded: Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}")
    
    # --- Update Input Size Logging ---
    new_input_size = X_train.shape[-1]
    print(f"Input feature size: {new_input_size}")

    # Store input size in config for model building
    cfg['input_size'] = new_input_size

    return train_loader, val_loader, test_loader, feat_scaler, targ_scaler

# Load the data
train_loader, val_loader, test_loader, feature_scaler, target_scaler = load_and_prepare_data(config)

Loading and preparing data...
Feature train_val shape: (28032, 14)
Training Data Sample:


,temperature_exterieure,humidite,ensoleillement,vitesse_vent,direction_vent,temperature_interieure,hour_sin,hour_cos,dayofweek,month_sin,month_cos,temp_diff_6hr,temp_roll_mean_6hr,temp_roll_std_6hr
0,3.25,82.00,0.0,6.400,210.0,21.051795,0.000000,1.000000,5.0,0.5,0.866025,0.0,0.000000,0.000000
1,1.50,83.00,0.0,4.600,220.0,20.999963,0.000000,1.000000,5.0,0.5,0.866025,0.0,0.000000,0.000000
2,-0.25,84.00,0.0,2.800,230.0,21.000107,0.000000,1.000000,5.0,0.5,0.866025,0.0,0.000000,0.000000
3,-2.00,85.00,0.0,1.000,240.0,21.000071,0.000000,1.000000,5.0,0.5,0.866025,0.0,0.000000,0.000000
4,-1.95,85.00,0.0,1.000,242.5,21.000084,0.258819,0.965926,5.0,0.5,0.866025,0.0,0.000000,0.000000
5,-1.90,85.00,0.0,1.000,245.0,21.000060,0.258819,0.965926,5.0,0.5,0.866025,0.0,-0.225000,2.190148
6,-1.85,85.00,0.0,1.000,247.5,21.000017,0.258819,0.965926,5.0,0.5,0.866025,-5.1,-1.075000,1.429248
7,-1.80,85.00,0.0,1.000,250.0,21.000111,0.258819,0.965926,5.0,0.5,0.866025,-3.3,-1.625000,0.677311
8,-1.85,86.75,0.0,1.275,245.0,21.000079,0.500000,0.866025,5.0,0.5,0.866025,-1.6,-1.891667,0.073598
9,-1.90,88.50,0.0,1.550,240.0,21.000152,0.500000,0.866025,5.0,0.5,0.866025,0.1,-1.875000,0.052440


Testing Data Sample:


,temperature_exterieure,humidite,ensoleillement,vitesse_vent,direction_vent,temperature_interieure,hour_sin,hour_cos,dayofweek,month_sin,month_cos,temp_diff_6hr,temp_roll_mean_6hr,temp_roll_std_6hr
0,10.75,77.5,0.0,9.225,42.5,20.965343,0.000000,1.000000,3.0,-0.866025,0.5,0.0,0.0,0.0
1,10.50,79.0,0.0,9.750,45.0,20.965533,0.000000,1.000000,3.0,-0.866025,0.5,0.0,0.0,0.0
2,10.25,80.5,0.0,10.275,47.5,20.967228,0.000000,1.000000,3.0,-0.866025,0.5,0.0,0.0,0.0
3,10.00,82.0,0.0,10.800,50.0,20.968734,0.000000,1.000000,3.0,-0.866025,0.5,0.0,0.0,0.0
4,9.95,82.0,0.0,10.925,50.0,20.991033,0.258819,0.965926,3.0,-0.866025,0.5,0.0,0.0,0.0


Data loaded: Train=23787, Val=4198, Test=6945
Input feature size: 14


## 3. Hyperparameter Optimization (Optuna)

Define the training and evaluation logic for a single trial, then use Optuna to search for the best hyperparameters.

In [78]:
def run_training_trial(trial_cfg, train_loader, val_loader, test_loader, device, trial_num=None, random_seed=None):
    """
    Runs a single training and evaluation trial.
    Saves the best model, scalers, and results.
    Returns the validation MAE (for Optuna), test MAE, and paths to saved artifacts.
    """
    # --- Setup & Seeding ---
    current_seed = random_seed if random_seed is not None else random.randint(1, 10000)
    set_seed(current_seed)
    trial_cfg['current_seed'] = current_seed  # Store the seed used

    # --- File Naming & Directories ---
    output_dir = trial_cfg['output_dir']
    results_dir = trial_cfg['results_dir']
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(results_dir, exist_ok=True)

    run_id = f"trial_{trial_num}_seed_{current_seed}" if trial_num is not None else f"seed_{current_seed}"

    model_filename = f"model_{run_id}.pth"
    feature_scaler_filename = f"feature_scaler_{run_id}.joblib"
    target_scaler_filename = f"target_scaler_{run_id}.joblib"
    results_filename = f"results_{run_id}.csv"

    model_path = os.path.join(output_dir, model_filename)
    feature_scaler_path = os.path.join(output_dir, feature_scaler_filename)
    target_scaler_path = os.path.join(output_dir, target_scaler_filename)
    results_path = os.path.join(results_dir, results_filename)

    # --- Build Model, Criterion, Optimizer, Scheduler ---
    model = LSTMModel(
        input_size=trial_cfg['input_size'],
        hidden_size=trial_cfg['hidden_size'],
        num_layers=trial_cfg['num_layers'],
        dropout=trial_cfg['dropout']
    ).to(device)

    criterion = nn.HuberLoss() if trial_cfg['loss_function'] == 'Huber' else nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=trial_cfg['lr'])

    scheduler = None
    if trial_cfg['use_lr_scheduler']:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=trial_cfg['lr_scheduler_factor'],
            patience=trial_cfg['lr_scheduler_patience'], verbose=False
        )

    # --- Training Loop ---
    best_val_loss = float('inf')
    epochs_no_improve = 0
    pbar_epochs = tqdm(range(1, trial_cfg['num_epochs'] + 1), desc=f'Run {run_id} Epochs', leave=False)

    for epoch in pbar_epochs:
        # Train
        model.train()
        train_loss = 0.0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(Xb)
            loss = criterion(out, yb)
            loss.backward()
            if trial_cfg['use_gradient_clipping']:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=trial_cfg['gradient_clip_max_norm'])
            optimizer.step()
            train_loss += loss.item() * Xb.size(0)
        avg_train_loss = train_loss / len(train_loader.dataset)

        # Validate
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                out = model(Xb)
                loss = criterion(out, yb)
                val_loss += loss.item() * Xb.size(0)
        avg_val_loss = val_loss / len(val_loader.dataset)

        pbar_epochs.set_postfix({'TrL': f'{avg_train_loss:.5f}', 'VlL': f'{avg_val_loss:.5f}'})

        if trial_cfg['use_lr_scheduler'] and scheduler:
            scheduler.step(avg_val_loss)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), model_path)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if trial_cfg['use_early_stopping'] and epochs_no_improve >= trial_cfg['early_stopping_patience']:
                print(f"Early stopping triggered after epoch {epoch} for run {run_id}")
                break

    # --- Evaluation on Test Set ---
    test_mae = float('inf')
    if os.path.exists(model_path):  # Only evaluate if a model was saved
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.eval()
        preds_scaled_tensors = [] # Changed variable name to reflect it holds tensors
        trues_scaled_list = [] # Renamed for consistency, holds numpy arrays from dataloader
        with torch.no_grad():
            for Xb, yb in test_loader:
                Xb = Xb.to(device)
                # Keep prediction tensor on the device (GPU)
                out_scaled_tensor = model(Xb)
                preds_scaled_tensors.append(out_scaled_tensor)
                # Assuming yb from DataLoader is already on CPU or small enough that .numpy() is ok
                trues_scaled_list.append(yb.numpy())

        # Concatenate tensors on the device, then move to CPU and convert to numpy *once*
        preds_scaled = torch.cat(preds_scaled_tensors).cpu().numpy()
        # Concatenate the numpy arrays
        trues_scaled = np.concatenate(trues_scaled_list)

        # Inverse transform remains the same
        preds = target_scaler.inverse_transform(preds_scaled).flatten()
        trues = target_scaler.inverse_transform(trues_scaled).flatten()

        test_mse = mean_squared_error(trues, preds)
        test_mae = mean_absolute_error(trues, preds)

        results_df = pd.DataFrame({'y_true': trues, 'y_pred': preds})
        results_df.to_csv(results_path, index=False)

        # Save scalers associated with the best model
        joblib.dump(feature_scaler, feature_scaler_path)
        joblib.dump(target_scaler, target_scaler_path)
        print(f"Run {run_id}: Best Val Loss: {best_val_loss:.5f}, Test MAE: {test_mae:.5f}. Artifacts saved.")

        # Prepare artifact info dictionary
        artifacts = {
            "model_path": model_path,
            "feature_scaler_path": feature_scaler_path,
            "target_scaler_path": target_scaler_path,
            "results_path": results_path,
            "val_loss": best_val_loss,
            "test_mae": test_mae,
            "params": trial_cfg
        }
        return best_val_loss, test_mae, artifacts
    else:
        print(f"Run {run_id}: No model saved (training might have failed or not improved).")
        return float('inf'), float('inf'), None


In [79]:
# --- Optuna Objective Function ---
def objective(trial, base_cfg, train_loader, val_loader, test_loader, device):
    """Optuna objective: suggest hyperparameters, run trial, return validation metric."""
    # Create a copy of the base config for this trial
    trial_cfg = base_cfg.copy()

    # Suggest hyperparameters
    trial_cfg['lr'] = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    trial_cfg['num_layers'] = trial.suggest_int("num_layers", 1, 4)
    trial_cfg['batch_size'] = trial.suggest_categorical("batch_size", [32, 64, 128])
    trial_cfg['hidden_size'] = trial.suggest_int("hidden_size", 32, 128, step=16)
    trial_cfg['dropout'] = trial.suggest_float("dropout", 0.0, 0.5)
    # Optional: trial_cfg['seq_len'] = trial.suggest_categorical("seq_len", [8, 16, 24])

    # Run the training trial
    val_metric, test_mae, artifacts = run_training_trial(
        trial_cfg, train_loader, val_loader, test_loader, device, trial_num=trial.number
    )

    # Store artifacts info in the trial for later retrieval
    if artifacts:
        trial.set_user_attr("artifacts", artifacts)
        trial.set_user_attr("test_mae", test_mae)  # Store test MAE for reference

    # Optuna minimizes the returned metric (validation loss)
    return val_metric

In [81]:
# --- Run Optuna Study ---
print("\n--- Starting Hyperparameter Optimization ---")

study = optuna.create_study(
    study_name=config['optuna_study_name'],
    storage=config['optuna_storage_url'],
    direction="minimize",
    load_if_exists=True  # Resume study if storage is configured and study exists
)

# Pass fixed arguments (loaders, device) using a lambda
study.optimize(
    lambda trial: objective(trial, config, train_loader, val_loader, test_loader, device),
    n_trials=config['n_trials']
)

# --- Get Best Trial Results ---
best_trial = study.best_trial
best_artifacts = None
if best_trial:
    print(f"\nOptimization Finished.")
    print(f"Best Trial Number: {best_trial.number}")
    print(f"Best Validation Metric (Loss): {best_trial.value:.5f}")
    print(f"Best Hyperparameters:")
    pprint.pprint(best_trial.params)

    if "artifacts" in best_trial.user_attrs:
        best_artifacts = best_trial.user_attrs["artifacts"]
        print(f"Best Trial Test MAE: {best_artifacts['test_mae']:.5f}")
        print(f"Model saved to: {best_artifacts['model_path']}")
    else:
        print("Warning: Artifacts not found for the best trial.")
else:
    print("Optimization did not complete successfully or found no best trial.")

# Store best hyperparameters in the main config for the next step
if best_artifacts:
    config.update(best_trial.params)  # Update config with best hyperparams
    config['best_trial_artifacts'] = best_artifacts  # Store artifact info
    print("\nUpdated config with best hyperparameters found.")

[I 2025-04-19 05:42:52,860] A new study created in memory with name: hvac_optimization_lstm_simplified



--- Starting Hyperparameter Optimization ---


[I 2025-04-19 05:43:06,186] Trial 0 finished with value: 0.0007958422687167338 and parameters: {'lr': 0.007366711040356899, 'num_layers': 4, 'batch_size': 32, 'hidden_size': 48, 'dropout': 0.013389094306605731}. Best is trial 0 with value: 0.0007958422687167338.
/home/ben/Documents/Projects/IFT3395-project/.venv310/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Early stopping triggered after epoch 14 for run trial_0_seed_689
Run trial_0_seed_689: Best Val Loss: 0.00080, Test MAE: 0.59237. Artifacts saved.


[I 2025-04-19 05:43:45,104] Trial 1 finished with value: 0.0008219400229551015 and parameters: {'lr': 4.748753239047673e-05, 'num_layers': 4, 'batch_size': 128, 'hidden_size': 128, 'dropout': 0.4877988843702916}. Best is trial 0 with value: 0.0007958422687167338.
/home/ben/Documents/Projects/IFT3395-project/.venv310/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Early stopping triggered after epoch 36 for run trial_1_seed_9970
Run trial_1_seed_9970: Best Val Loss: 0.00082, Test MAE: 0.65294. Artifacts saved.


... (4 outputs left out)

[I 2025-04-19 05:44:51,848] Trial 4 finished with value: 0.0010432935993818513 and parameters: {'lr': 6.0756041865042614e-05, 'num_layers': 3, 'batch_size': 64, 'hidden_size': 128, 'dropout': 0.34478159670628894}. Best is trial 3 with value: 0.0007717035471602809.
/home/ben/Documents/Projects/IFT3395-project/.venv310/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Early stopping triggered after epoch 33 for run trial_4_seed_6118
Run trial_4_seed_6118: Best Val Loss: 0.00104, Test MAE: 0.66845. Artifacts saved.


[W 2025-04-19 05:45:06,502] Trial 5 failed with parameters: {'lr': 5.1844969476340075e-05, 'num_layers': 4, 'batch_size': 64, 'hidden_size': 80, 'dropout': 0.001888853814377589} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/ben/Documents/Projects/IFT3395-project/.venv310/lib/python3.10/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_4198/3280548091.py", line 13, in <lambda>
    lambda trial: objective(trial, config, train_loader, val_loader, test_loader, device),
  File "/tmp/ipykernel_4198/2972314180.py", line 16, in objective
    val_metric, test_mae, artifacts = run_training_trial(
  File "/tmp/ipykernel_4198/263863547.py", line 58, in run_training_trial
    Xb, yb = Xb.to(device), yb.to(device)
KeyboardInterrupt
[W 2025-04-19 05:45:06,503] Trial 5 failed with value None.


KeyboardInterrupt: 

## 4. Ensemble Training (Bagging)

Train multiple models using the best hyperparameters identified by Optuna, each initialized with a different random seed. Store the artifact paths for each model.

In [47]:
def run_ensemble_training(base_cfg, seeds, train_loader, val_loader, test_loader, device):
    """Trains multiple models with the same config but different seeds."""
    print(f"\n--- Starting Ensemble Training for Seeds: {seeds} ---")
    ensemble_artifacts = {}

    # Ensure best params are used if available, otherwise use defaults
    if 'best_trial_artifacts' in base_cfg:
        print("Using hyperparameters from the best optimization trial.")
        # Create a config for ensemble runs based on best params
        ensemble_cfg = base_cfg['best_trial_artifacts']['params'].copy()
        # Ensure essential paths/settings from base_cfg are kept
        ensemble_cfg.update({k: v for k, v in base_cfg.items() if k not in ensemble_cfg})
    else:
        print("Warning: Best trial artifacts not found. Using default config for ensemble.")
        ensemble_cfg = base_cfg.copy()

    for i, seed in enumerate(seeds):
        print(f"\nTraining ensemble member {i+1}/{len(seeds)} with seed {seed}...")
        # Ensure each run has a unique seed but uses the same hyperparameters
        member_cfg = ensemble_cfg.copy()
        
        # Run training trial for this seed
        val_metric, test_mae, artifacts = run_training_trial(
            member_cfg, train_loader, val_loader, test_loader, device, trial_num=f"ensemble_{i}", random_seed=seed
        )
        
        if artifacts:
            ensemble_artifacts[f"seed_{seed}"] = artifacts
            print(f"Ensemble member {i+1} (Seed {seed}) finished. Test MAE: {test_mae:.5f}")
        else:
            print(f"Ensemble member {i+1} (Seed {seed}) failed to train successfully.")
            
    # Save the collected artifact paths
    output_dir = base_cfg['output_dir']
    os.makedirs(output_dir, exist_ok=True)
    artifacts_path = os.path.join(output_dir, base_cfg['ensemble_artifacts_file'])
    with open(artifacts_path, 'w') as f:
        # Use a custom serializer for Namespace if needed, or convert params to dict
        def serialize_artifacts(artifacts_dict):
            serializable = {}
            for key, data in artifacts_dict.items():
                if isinstance(data['params'], argparse.Namespace):
                    data['params'] = vars(data['params'])
                serializable[key] = data
            return serializable
        json.dump(serialize_artifacts(ensemble_artifacts), f, indent=4)
        
    print(f"\nEnsemble training finished. Artifact details saved to: {artifacts_path}")
    return ensemble_artifacts

# Run Ensemble Training
ensemble_results = run_ensemble_training(
    config, 
    config['ensemble_seeds'], 
    train_loader, 
    val_loader, 
    test_loader, 
    device
)

/home/ben/Documents/Projects/IFT3395-project/.venv310/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(



--- Starting Ensemble Training for Seeds: [42, 123, 456, 789, 1011] ---
Using hyperparameters from the best optimization trial.

Training ensemble member 1/5 with seed 42...


/home/ben/Documents/Projects/IFT3395-project/.venv310/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Run trial_ensemble_0_seed_42: Best Val Loss: 0.00038, Test MAE: 0.64942. Artifacts saved.
Ensemble member 1 (Seed 42) finished. Test MAE: 0.64942

Training ensemble member 2/5 with seed 123...


/home/ben/Documents/Projects/IFT3395-project/.venv310/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Early stopping triggered after epoch 32 for run trial_ensemble_1_seed_123
Run trial_ensemble_1_seed_123: Best Val Loss: 0.00039, Test MAE: 0.63039. Artifacts saved.
Ensemble member 2 (Seed 123) finished. Test MAE: 0.63039

Training ensemble member 3/5 with seed 456...


/home/ben/Documents/Projects/IFT3395-project/.venv310/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Early stopping triggered after epoch 36 for run trial_ensemble_2_seed_456
Run trial_ensemble_2_seed_456: Best Val Loss: 0.00044, Test MAE: 0.75262. Artifacts saved.
Ensemble member 3 (Seed 456) finished. Test MAE: 0.75262

Training ensemble member 4/5 with seed 789...


/home/ben/Documents/Projects/IFT3395-project/.venv310/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Early stopping triggered after epoch 46 for run trial_ensemble_3_seed_789
Run trial_ensemble_3_seed_789: Best Val Loss: 0.00049, Test MAE: 0.71794. Artifacts saved.
Ensemble member 4 (Seed 789) finished. Test MAE: 0.71794

Training ensemble member 5/5 with seed 1011...


Early stopping triggered after epoch 41 for run trial_ensemble_4_seed_1011
Run trial_ensemble_4_seed_1011: Best Val Loss: 0.00032, Test MAE: 0.61493. Artifacts saved.
Ensemble member 5 (Seed 1011) finished. Test MAE: 0.61493

Ensemble training finished. Artifact details saved to: models_lstm_simplified/ensemble_artifacts.json


## 5. Ensemble Prediction

Load the models trained during the ensemble phase, generate predictions on the final test dataset (`test_features.csv`), average the predictions, and save the submission file.

In [48]:
def prepare_prediction_data(cfg, feature_scaler_path):
    """Loads and prepares the final test data for prediction."""
    print(f"\nLoading prediction features from: {cfg['predict_features_csv']}")
    df_pred = pd.read_csv(cfg['predict_features_csv'], parse_dates=['date'])
    ids = df_pred['ID'].values

    # --- Feature Engineering ---
    all_feature_cols = list(cfg['base_feature_cols'])
    if cfg['use_time_features']:
        df_pred = add_time_features(df_pred)
        time_feature_cols = ['hour_sin', 'hour_cos', 'dayofweek', 'month_sin', 'month_cos']
        all_feature_cols.extend(time_feature_cols)

    # Ensure columns match training
    features = df_pred[all_feature_cols].values 

    # --- Load Scaler and Scale ---
    try:
        feat_scaler = joblib.load(feature_scaler_path)
    except FileNotFoundError:
        print(f"Error: Feature scaler not found at {feature_scaler_path}")
        return None, None
    features_scaled = feat_scaler.transform(features)

    # --- Create Sequences (Simplified for Prediction) ---
    # For prediction, we often just need the latest features to predict the future point.
    # Here, we create sequences by tiling the last available feature vector.
    num_samples, num_features = features_scaled.shape
    seq_len = cfg['seq_len']
    X_pred_seq = np.zeros((num_samples, seq_len, num_features), dtype=np.float32)
    for i in range(num_samples):
        X_pred_seq[i, :, :] = np.tile(features_scaled[i, :], (seq_len, 1))
        
    print(f"Prediction data prepared: {X_pred_seq.shape[0]} samples.")
    return X_pred_seq, ids

In [ ]:
def run_ensemble_prediction(pred_cfg, ensemble_artifacts, device):
    """Generates predictions using the ensemble and averages them."""
    print(f"\n--- Running Ensemble Prediction ---")
    if not ensemble_artifacts:
        print("Error: No ensemble artifacts provided. Cannot predict.")
        return None

    all_predictions = []
    final_ids = None
    target_scaler = None  # Load the first target scaler

    # Load the first member's artifacts to prepare data
    first_member_id = next(iter(ensemble_artifacts))
    first_member_data = ensemble_artifacts[first_member_id]
    feature_scaler_path = first_member_data['feature_scaler_path']
    target_scaler_path = first_member_data['target_scaler_path']
    member_params = first_member_data['params']  # Use params from a member

    # Merge member params with base prediction config if needed (e.g., for seq_len)
    effective_cfg = pred_cfg.copy()
    effective_cfg.update(member_params)

    # Prepare prediction data using the first member's feature scaler
    X_pred_seq, final_ids = prepare_prediction_data(effective_cfg, feature_scaler_path)
    if X_pred_seq is None:
        return None
        
    # Load the target scaler (assume they are compatible across ensemble)
    try:
        target_scaler = joblib.load(target_scaler_path)
    except FileNotFoundError:
        print(f"Error: Target scaler not found at {target_scaler_path}")
        return None

    # Iterate through each ensemble member for prediction
    for member_id, member_data in tqdm(ensemble_artifacts.items(), desc="Ensemble Members"):
        model_path = member_data['model_path']
        member_params = member_data['params']  # Config used to train this member
        
        try:
            # Build model architecture based on this member's parameters
            model = LSTMModel(
                input_size=member_params['input_size'],
                hidden_size=member_params['hidden_size'],
                num_layers=member_params['num_layers'],
                dropout=member_params['dropout']
            ).to(device)
            model.load_state_dict(torch.load(model_path, map_location=device))
            model.eval()

            # Predict using batches
            preds_scaled_list = []
            pred_dataset = TensorDataset(torch.from_numpy(X_pred_seq).float())
            pred_loader = DataLoader(pred_dataset, batch_size=pred_cfg.get('batch_size', 128), shuffle=False)

            with torch.no_grad():
                for Xb, in pred_loader:
                    Xb = Xb.to(device)
                    batch_preds_scaled = model(Xb).to(device).cpu().numpy()
                    preds_scaled_list.append(batch_preds_scaled)
            
            preds_scaled = np.concatenate(preds_scaled_list)
            preds = target_scaler.inverse_transform(preds_scaled).flatten()
            all_predictions.append(preds)

        except Exception as e:
            print(f"Error processing member {member_id}: {e}")

    # Average predictions
    if not all_predictions:
        print("Error: No predictions generated from any ensemble member.")
        return None
        
    print(f"\nAveraging predictions from {len(all_predictions)} members.")
    final_preds = np.mean(np.vstack(all_predictions), axis=0)

    # Save final submission
    results_dir = pred_cfg['results_dir']
    os.makedirs(results_dir, exist_ok=True)
    submission_path = os.path.join(results_dir, pred_cfg['final_submission_filename'])
    df_out = pd.DataFrame({'ID': final_ids, 'puissance_cvac_future': final_preds})
    df_out.to_csv(submission_path, index=False)
    print(f"Ensemble prediction finished. Submission file saved to: {submission_path}")
    return submission_path

# Run Ensemble Prediction
if ensemble_results:  # Check if ensemble training produced results
    submission_file_path = run_ensemble_prediction(config, ensemble_results, device)
else:
    print("Skipping ensemble prediction as ensemble training did not produce results.")


--- Running Ensemble Prediction ---

Loading prediction features from: dataset/test_features.csv
Error: Feature scaler not found at models_lstm_simplified/feature_scaler_trial_ensemble_0_seed_42.joblib
